In [6]:
import pandas as pd
import numpy as np
import glob
import os
import time
import warnings

warnings.filterwarnings('ignore')

def parse_complex(val):
    """Converts Matlab format '0.5+0.2i' to Python format '0.5+0.2j'"""
    if isinstance(val, str):
        return complex(val.replace(' ', '').replace('i', 'j'))
    return val

def build_spatial_dataset(folder_path):
    all_features = []
    all_labels = []
    
    file_pattern = os.path.join(folder_path, '*.csv')
    file_list = glob.glob(file_pattern)
    
    print(f"Found {len(file_list)} CSV files. Building static spatial dataset...\n")
    
    for idx, file in enumerate(file_list):
        base_name = os.path.basename(file).replace('.csv', '').replace('o', '') 
        occupancy_count = base_name.count('1')
        
        try:
            # 1. Load the complex CSI text matrix
            df_raw = pd.read_csv(file, header=None)
            
            # 2. Parse numbers into Python Complex Floats
            if hasattr(df_raw, 'map'):
                df_complex = df_raw.map(parse_complex)
            else:
                df_complex = df_raw.applymap(parse_complex)
                
            # 3. For Static Sensing, the amplitudes of ALL subcarriers represent the "fingerprint" of the room
            # This yields a matrix of shape (50 measurements, N subcarriers)
            amplitude_matrix = np.abs(df_complex.values)
            
            # 4. Label every measurement in this file with the correct occupancy count
            labels = np.full(amplitude_matrix.shape[0], occupancy_count)
            
            all_features.append(amplitude_matrix)
            all_labels.append(labels)
            
            print(f"[{idx+1}/{len(file_list)}] {os.path.basename(file):<10} -> Ground Truth: {occupancy_count} people | Shape: {amplitude_matrix.shape}")
            
        except Exception as e:
            print(f"  -> Error on {base_name}: {e}")
            continue

    X_matrix = np.vstack(all_features)
    y_vector = np.concatenate(all_labels)
    
    return X_matrix, y_vector

# ==========================================
# EXECUTE & SAVE
# ==========================================
print("--- RUNNING CROWD COUNTING PIPELINE ---")
start = time.time()

# Target our new inside-AP dataset
X, y = build_spatial_dataset('Multiband WiFi Passive sensing/5 GHz AP in')

# Dynamically name the columns based on the number of subcarriers (e.g., Subcarrier_0, Subcarrier_1...)
columns = [f"Subcarrier_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=columns)

# Append Output Label
df['Occupancy_Count'] = y

csv_filename = 'Multiband_Crowd_Counting.csv'
df.to_csv(csv_filename, index=False)

print("\n--- DATASET GENERATION COMPLETE ---")
print(f"Time Taken: {time.time() - start:.2f} seconds")
print(f"Total Rows (Snapshots extracted): {df.shape[0]}")
print(f"Total Features (Subcarriers): {df.shape[1] - 1}") # -1 because the last column is the Label
print("\nClass Balance (Number of People):")
print(df['Occupancy_Count'].value_counts().sort_index())
display(df.head())

--- RUNNING CROWD COUNTING PIPELINE ---
Found 16 CSV files. Building static spatial dataset...

[1/16] 0000.csv   -> Ground Truth: 0 people | Shape: (52, 500)
[2/16] 0001.csv   -> Ground Truth: 1 people | Shape: (52, 500)
[3/16] 0010.csv   -> Ground Truth: 1 people | Shape: (52, 500)
[4/16] 0011.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[5/16] 0100.csv   -> Ground Truth: 1 people | Shape: (52, 500)
[6/16] 0101.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[7/16] 0110.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[8/16] 0111.csv   -> Ground Truth: 3 people | Shape: (52, 500)
[9/16] 1000.csv   -> Ground Truth: 1 people | Shape: (52, 500)
[10/16] 1001.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[11/16] 1010.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[12/16] 1011.csv   -> Ground Truth: 3 people | Shape: (52, 500)
[13/16] 1100.csv   -> Ground Truth: 2 people | Shape: (52, 500)
[14/16] 1101.csv   -> Ground Truth: 3 people | Shape: (52, 500)
[15/16] 1110.csv 

,Subcarrier_0,Subcarrier_1,Subcarrier_2,Subcarrier_3,Subcarrier_4,Subcarrier_5,Subcarrier_6,Subcarrier_7,Subcarrier_8,Subcarrier_9,...,Subcarrier_491,Subcarrier_492,Subcarrier_493,Subcarrier_494,Subcarrier_495,Subcarrier_496,Subcarrier_497,Subcarrier_498,Subcarrier_499,Occupancy_Count
0,1.373794,1.353273,1.544406,1.134543,1.252932,1.411409,1.336443,1.584556,1.384619,1.200162,...,1.162674,1.189696,1.262666,1.187003,1.177455,1.156096,1.204609,1.095442,1.295895,0
1,1.421603,1.278935,1.563887,1.197798,1.367902,1.306622,1.430038,1.231048,1.463235,1.332301,...,1.169037,1.439103,1.271204,1.207313,0.973291,1.223068,1.137903,1.227278,1.159102,0
2,1.431787,1.388116,1.344992,1.340666,1.237426,1.319371,1.174197,1.382425,1.406901,1.316152,...,1.070433,1.217441,1.237920,1.250567,0.934615,1.145027,1.049458,1.090291,1.094062,0
3,1.446550,1.431979,1.375361,1.375329,1.442654,1.278572,1.317843,1.334790,1.369315,1.504254,...,1.094153,1.277793,1.067017,1.231970,1.216773,1.227959,1.093635,1.027562,0.994303,0
4,1.272390,1.157515,1.220794,1.038379,1.308348,1.202234,1.170191,1.287532,0.973875,1.064063,...,1.041075,1.153613,1.186753,1.045935,0.890803,1.132599,1.027970,0.748251,1.046802,0


In [7]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

print("--- INITIALIZING MULTI-CLASS ML PIPELINE ---")

X = df.drop('Occupancy_Count', axis=1)
y = df['Occupancy_Count']

# 1. Stratified Split (Ensures test set has the same bell-curve distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scale the 500 subcarriers
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original Training Class Balance:")
print(y_train.value_counts().sort_index().to_dict())

# 3. Apply SMOTE to balance the extreme classes (0 and 4)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("\nBalanced SMOTE Training Class Balance:")
print(y_train_smote.value_counts().sort_index().to_dict())

results = []

# ==========================================
# 1. Random Forest
# ==========================================
print("\nTraining Random Forest...")
rf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
start_time = time.time()
rf.fit(X_train_smote, y_train_smote)
rf_preds = rf.predict(X_test_scaled)
results.append({
    "Model": "Random Forest", 
    "Accuracy": accuracy_score(y_test, rf_preds), 
    "Macro F1-Score": f1_score(y_test, rf_preds, average='macro'), 
    "Time (s)": time.time() - start_time
})

# ==========================================
# 2. XGBoost
# ==========================================
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
start_time = time.time()
xgb.fit(X_train_smote, y_train_smote)
xgb_preds = xgb.predict(X_test_scaled)
results.append({
    "Model": "XGBoost", 
    "Accuracy": accuracy_score(y_test, xgb_preds), 
    "Macro F1-Score": f1_score(y_test, xgb_preds, average='macro'),
    "Time (s)": time.time() - start_time
})

# ==========================================
# 3. LightGBM
# ==========================================
print("Training LightGBM...")
lgbm = LGBMClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1)
start_time = time.time()
lgbm.fit(X_train_smote, y_train_smote)
lgbm_preds = lgbm.predict(X_test_scaled)
results.append({
    "Model": "LightGBM", 
    "Accuracy": accuracy_score(y_test, lgbm_preds), 
    "Macro F1-Score": f1_score(y_test, lgbm_preds, average='macro'),
    "Time (s)": time.time() - start_time
})

# ==========================================
# THE RESULTS
# ==========================================
print("\n" + "="*70)
print("🏆 MULTI-CLASS CROWD COUNTING RESULTS (500 SPATIAL FEATURES) 🏆")
print("="*70)

results_df = pd.DataFrame(results).sort_values(by="Macro F1-Score", ascending=False)

print(f"{'Model':<15} | {'Accuracy':<10} | {'Macro F1':<10} | {'Train Time (s)':<15}")
print("-" * 70)
for index, row in results_df.iterrows():
    print(f"{row['Model']:<15} | {row['Accuracy']*100:>9.2f}% | {row['Macro F1-Score']*100:>9.2f}% | {row['Time (s)']:>13.4f}s")
print("="*70)

print("\n--- DETAILED XGBOOST REPORT ---")
# This will show you exactly how well the model predicts each individual count (0, 1, 2, 3, 4)
print(classification_report(y_test, xgb_preds))

--- INITIALIZING MULTI-CLASS ML PIPELINE ---
Original Training Class Balance:
{0: 42, 1: 166, 2: 249, 3: 166, 4: 42}

Balanced SMOTE Training Class Balance:
{0: 249, 1: 249, 2: 249, 3: 249, 4: 249}

Training Random Forest...
Training XGBoost...
Training LightGBM...

🏆 MULTI-CLASS CROWD COUNTING RESULTS (500 SPATIAL FEATURES) 🏆
Model           | Accuracy   | Macro F1   | Train Time (s) 
----------------------------------------------------------------------
Random Forest   |     63.47% |     62.53% |        0.3241s
LightGBM        |     62.87% |     57.51% |        1.8700s
XGBoost         |     62.87% |     56.07% |        6.6567s

--- DETAILED XGBOOST REPORT ---
              precision    recall  f1-score   support

           0       0.50      0.10      0.17        10
           1       0.57      0.64      0.61        42
           2       0.63      0.81      0.71        63
           3       0.67      0.43      0.52        42
           4       0.80      0.80      0.80        10

    

In [8]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

print("--- INITIALIZING PCA MULTI-CLASS PIPELINE ---")

X = df.drop('Occupancy_Count', axis=1)
y = df['Occupancy_Count']

# 1. Stratified Split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scale the data (PCA is highly sensitive to variance, so scaling is mandatory)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Apply PCA 
# We'll ask it to keep enough components to explain 95% of the variance in the room
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA reduced 500 subcarriers down to -> {X_train_pca.shape[1]} core components!\n")

# 4. Apply SMOTE to the PCA-reduced data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_pca, y_train)

results = []

# ==========================================
# 1. Random Forest (PCA Version)
# ==========================================
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
start_time = time.time()
rf.fit(X_train_smote, y_train_smote)
rf_preds = rf.predict(X_test_pca)
results.append({
    "Model": "Random Forest (PCA)", 
    "Accuracy": accuracy_score(y_test, rf_preds), 
    "Macro F1-Score": f1_score(y_test, rf_preds, average='macro'), 
    "Time (s)": time.time() - start_time
})

# ==========================================
# 2. XGBoost (PCA Version)
# ==========================================
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
start_time = time.time()
xgb.fit(X_train_smote, y_train_smote)
xgb_preds = xgb.predict(X_test_pca)
results.append({
    "Model": "XGBoost (PCA)", 
    "Accuracy": accuracy_score(y_test, xgb_preds), 
    "Macro F1-Score": f1_score(y_test, xgb_preds, average='macro'),
    "Time (s)": time.time() - start_time
})

# ==========================================
# 3. LightGBM (PCA Version)
# ==========================================
print("Training LightGBM...")
lgbm = LGBMClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1)
start_time = time.time()
lgbm.fit(X_train_smote, y_train_smote)
lgbm_preds = lgbm.predict(X_test_pca)
results.append({
    "Model": "LightGBM (PCA)", 
    "Accuracy": accuracy_score(y_test, lgbm_preds), 
    "Macro F1-Score": f1_score(y_test, lgbm_preds, average='macro'),
    "Time (s)": time.time() - start_time
})

# ==========================================
# THE RESULTS
# ==========================================
print("\n" + "="*70)
print("🏆 PCA OPTIMIZED CROWD COUNTING RESULTS 🏆")
print("="*70)

results_df = pd.DataFrame(results).sort_values(by="Macro F1-Score", ascending=False)

print(f"{'Model':<20} | {'Accuracy':<10} | {'Macro F1':<10} | {'Train Time (s)':<15}")
print("-" * 70)
for index, row in results_df.iterrows():
    print(f"{row['Model']:<20} | {row['Accuracy']*100:>9.2f}% | {row['Macro F1-Score']*100:>9.2f}% | {row['Time (s)']:>13.4f}s")
print("="*70)

print("\n--- DETAILED RANDOM FOREST (PCA) REPORT ---")
# Random forest usually handles PCA components slightly better than un-tuned Gradient Boosting
print(classification_report(y_test, rf_preds))

--- INITIALIZING PCA MULTI-CLASS PIPELINE ---
PCA reduced 500 subcarriers down to -> 94 core components!

Training Random Forest...
Training XGBoost...
Training LightGBM...

🏆 PCA OPTIMIZED CROWD COUNTING RESULTS 🏆
Model                | Accuracy   | Macro F1   | Train Time (s) 
----------------------------------------------------------------------
LightGBM (PCA)       |     88.02% |     88.73% |        0.3641s
Random Forest (PCA)  |     88.62% |     87.81% |        0.2350s
XGBoost (PCA)        |     86.83% |     85.75% |        0.9934s

--- DETAILED RANDOM FOREST (PCA) REPORT ---
              precision    recall  f1-score   support

           0       0.67      0.80      0.73        10
           1       0.88      0.90      0.89        42
           2       0.90      0.90      0.90        63
           3       0.90      0.83      0.86        42
           4       1.00      1.00      1.00        10

    accuracy                           0.89       167
   macro avg       0.87      0.8

In [9]:
import time
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- INITIATING LIGHTGBM HYPERPARAMETER GRID SEARCH ---")

# Define the hyperparameter space to explore
param_dist = {
    'n_estimators': randint(100, 500),         # Number of boosting rounds
    'learning_rate': uniform(0.01, 0.2),       # Step size shrinkage
    'max_depth': randint(3, 15),               # Maximum tree depth
    'num_leaves': randint(20, 150),            # Max leaves per tree
    'subsample': uniform(0.6, 0.4),            # Fraction of data to sample
    'colsample_bytree': uniform(0.6, 0.4)      # Feature fraction per tree
}

# Initialize the base model (we use the PCA & SMOTE data from the previous cell)
# Note: 'X_train_smote' and 'y_train_smote' are already loaded in memory
lgbm_tuner = LGBMClassifier(random_state=42, verbose=-1)

# Set up the Randomized Search (Testing 20 combinations across 3 folds)
random_search = RandomizedSearchCV(
    estimator=lgbm_tuner,
    param_distributions=param_dist,
    n_iter=20,                 
    cv=3,                      
    scoring='f1_macro',        
    n_jobs=-1,                 
    random_state=42,
    verbose=1
)

print("Searching for the optimal architecture (this may take a minute)...")
start_time = time.time()
random_search.fit(X_train_smote, y_train_smote)
tuning_time = time.time() - start_time

# Extract the absolute best model found
best_lgbm = random_search.best_estimator_

# Test the optimized model on the unseen Test set
best_preds = best_lgbm.predict(X_test_pca)
final_acc = accuracy_score(y_test, best_preds)

print("\n" + "="*70)
print("🚀 HYPERPARAMETER TUNING COMPLETE 🚀")
print("="*70)
print(f"Tuning Time: {tuning_time:.2f} seconds")
print(f"Optimal Parameters Found:")
for k, v in random_search.best_params_.items():
    if isinstance(v, float):
        print(f"  * {k}: {v:.4f}")
    else:
        print(f"  * {k}: {v}")

print("-" * 70)
print(f"🔥 FINAL TUNED ACCURACY: {final_acc*100:.2f}% 🔥")
print("-" * 70)

print("\nDetailed Tuned Report:")
print(classification_report(y_test, best_preds))

# A quick look at where the model is making its mistakes
print("\nConfusion Matrix (Rows=True, Cols=Predicted):")
print(confusion_matrix(y_test, best_preds))

--- INITIATING LIGHTGBM HYPERPARAMETER GRID SEARCH ---
Searching for the optimal architecture (this may take a minute)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

🚀 HYPERPARAMETER TUNING COMPLETE 🚀
Tuning Time: 50.50 seconds
Optimal Parameters Found:
  * colsample_bytree: 0.7757
  * learning_rate: 0.0503
  * max_depth: 9
  * n_estimators: 340
  * num_leaves: 71
  * subsample: 0.8253
----------------------------------------------------------------------
🔥 FINAL TUNED ACCURACY: 89.82% 🔥
----------------------------------------------------------------------

Detailed Tuned Report:
              precision    recall  f1-score   support

           0       0.69      0.90      0.78        10
           1       0.83      0.93      0.88        42
           2       0.95      0.90      0.93        63
           3       0.95      0.83      0.89        42
           4       1.00      1.00      1.00        10

    accuracy                           0.90       167
   macro avg  

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

print("--- EXTRACTING STATISTICAL FEATURES & PCA ---")

# Assuming 'df' is still in memory from the first cell
X_raw = df.drop('Occupancy_Count', axis=1)
y = df['Occupancy_Count']

# 1. Feature Engineering: Calculate row-wise statistics across the 500 subcarriers
print("Calculating statistical features across subcarriers...")
X_stats = pd.DataFrame()
X_stats['mean'] = X_raw.mean(axis=1)
X_stats['std'] = X_raw.std(axis=1)
X_stats['var'] = X_raw.var(axis=1)
X_stats['max'] = X_raw.max(axis=1)
X_stats['min'] = X_raw.min(axis=1)
X_stats['ptp'] = X_stats['max'] - X_stats['min']

# skewness & kurtosis give hints about reflections heavily distorting extreme frequencies
X_stats['skew'] = X_raw.skew(axis=1)
X_stats['kurt'] = X_raw.kurtosis(axis=1)

# 2. Train/Test Split (stratify to preserve bell curve ratio)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)

# Split the new statistics using the same seed to match the rows
stats_train, stats_test = train_test_split(X_stats, test_size=0.2, random_state=42, stratify=y)

# 3. PCA on the raw subcarriers (as before)
print("Applying PCA on raw subcarriers...")
scaler_raw = StandardScaler()
X_train_raw_scaled = scaler_raw.fit_transform(X_train_raw)
X_test_raw_scaled = scaler_raw.transform(X_test_raw)

pca = PCA(n_components=0.95, random_state=42)
pca_train = pca.fit_transform(X_train_raw_scaled)
pca_test = pca.transform(X_test_raw_scaled)

# 4. Scale the statistical features separately
print("Scaling statistical features...")
scaler_stats = StandardScaler()
stats_train_scaled = scaler_stats.fit_transform(stats_train)
stats_test_scaled = scaler_stats.transform(stats_test)

# 5. Concatenate PCA components and Statistical Features
X_train_combined = np.hstack((pca_train, stats_train_scaled))
X_test_combined = np.hstack((pca_test, stats_test_scaled))
print(f"Final combined feature shape: {X_train_combined.shape[1]} features (PCA + Stats)\n")

# 6. Apply SMOTE to the combined feature set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_combined, y_train)

# 7. Train LightGBM using the optimal parameters found from Grid Search
print("Training Optimized LightGBM on Enriched Features...")
optim_lgbm = LGBMClassifier(
    n_estimators=340,
    learning_rate=0.0503,
    max_depth=9,
    num_leaves=71,
    subsample=0.8253,
    colsample_bytree=0.7757,
    random_state=42,
    verbose=-1
)

start_time = time.time()
optim_lgbm.fit(X_train_smote, y_train_smote)
train_time = time.time() - start_time

# 8. Evaluate!
preds = optim_lgbm.predict(X_test_combined)
final_acc = accuracy_score(y_test, preds)

print("\n" + "="*70)
print("ENRICHED FEATURE (PCA + STATS) RESULTS ")
print("="*70)
print(f"Train Time: {train_time:.4f} seconds")
print("-" * 70)
print(f" FINAL ACCURACY: {final_acc*100:.2f}% ")
print("-" * 70)

print("\nDetailed Report:")
print(classification_report(y_test, preds))

print("\nConfusion Matrix (Rows=True, Cols=Predicted):")
print(confusion_matrix(y_test, preds))

--- EXTRACTING STATISTICAL FEATURES & PCA ---
Calculating statistical features across subcarriers...
Applying PCA on raw subcarriers...
Scaling statistical features...
Final combined feature shape: 102 features (PCA + Stats)

Training Optimized LightGBM on Enriched Features...

🚀 ENRICHED FEATURE (PCA + STATS) RESULTS 🚀
Train Time: 1.3275 seconds
----------------------------------------------------------------------
🔥 FINAL ACCURACY: 91.02% 🔥
----------------------------------------------------------------------

Detailed Report:
              precision    recall  f1-score   support

           0       0.69      0.90      0.78        10
           1       0.89      0.93      0.91        42
           2       0.95      0.90      0.93        63
           3       0.95      0.88      0.91        42
           4       0.91      1.00      0.95        10

    accuracy                           0.91       167
   macro avg       0.88      0.92      0.90       167
weighted avg       0.92      0

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("--- ANALYZING FEATURE IMPORTANCE FOR REAL-TIME DEPLOYMENT ---")

# 1. Map names to our 102 features
pca_names = [f"PCA_{i+1}" for i in range(pca_train.shape[1])]
stat_names = ['Mean', 'Std', 'Variance', 'Max', 'Min', 'Peak-to-Peak', 'Skewness', 'Kurtosis']
all_feature_names = pca_names + stat_names

# 2. Extract importances from our highly-tuned LightGBM model
importances = optim_lgbm.feature_importances_

feature_imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Display the Top 15 Most Important Features
print("\nTop 15 Most Crucial Features for the Router:")
display(feature_imp_df.head(15))

# Calculate how many features we could theoretically drop
zero_importance = feature_imp_df[feature_imp_df['Importance'] == 0].shape[0]
print(f"\nWe can safely delete {zero_importance} completely useless features to speed up the router's real-time inference!")

# ==========================================
# 🚀 ADAPTIVE WI-FI MANAGER SIMULATION 🚀
# ==========================================
print("\n\n" + "="*70)
print("📡 STARTING ADAPTIVE WI-FI MANAGER SIMULATION 📡")
print("="*70)

def smart_router_manager(occupied_count):
    """Simulates firmware adjustments in a smart router based on room occupancy."""
    if occupied_count == 0:
        return "STATUS: EMPTY  -> ACTION: Reduced Tx Power to 20%. Entering Sleep Mode."
    elif occupied_count == 1:
        return "STATUS: 1 USER -> ACTION: Standard Tx Power (50%). Focusing beamforming on single target."
    elif occupied_count == 2:
        return "STATUS: 2 USERS -> ACTION: Standard Tx Power (75%). Splitting spatial streams."
    elif occupied_count >= 3:
        return "STATUS: CROWDED -> ACTION: MAX Tx Power (100%). MU-MIMO Activated to prevent bottlenecks."

# Let's simulate a 10-second real-time window by randomly picking 10 test samples
simulated_window = np.random.choice(len(preds), size=10, replace=False)

print("Simulating real-time CSI extraction and router adjustment over a 10-second window...\n")
for i, idx in enumerate(simulated_window, 1):
    actual_people = y_test.iloc[idx]
    predicted_people = preds[idx]
    
    # We pass the PREDICTED value to the router (because in the real world, the router doesn't know the actual count)
    router_action = smart_router_manager(predicted_people)
    
    # Check if the router made a mistake
    status_flag = "✅ Correct" if actual_people == predicted_people else f"❌ Error (Actual: {actual_people})"
    
    print(f"Sec {i:02d} | CSI Processed -> Predicted: {predicted_people} | {router_action} | {status_flag}")


--- ANALYZING FEATURE IMPORTANCE FOR REAL-TIME DEPLOYMENT ---

Top 15 Most Crucial Features for the Router:


,Feature,Importance
95,Std,1190
4,PCA_5,1119
5,PCA_6,1008
1,PCA_2,982
8,PCA_9,950
6,PCA_7,852
96,Variance,825
7,PCA_8,821
23,PCA_24,791
14,PCA_15,778



We can safely delete 0 completely useless features to speed up the router's real-time inference!


📡 STARTING ADAPTIVE WI-FI MANAGER SIMULATION 📡
Simulating real-time CSI extraction and router adjustment over a 10-second window...

Sec 01 | CSI Processed -> Predicted: 1 | STATUS: 1 USER -> ACTION: Standard Tx Power (50%). Focusing beamforming on single target. | ✅ Correct
Sec 02 | CSI Processed -> Predicted: 2 | STATUS: 2 USERS -> ACTION: Standard Tx Power (75%). Splitting spatial streams. | ✅ Correct
Sec 03 | CSI Processed -> Predicted: 3 | STATUS: CROWDED -> ACTION: MAX Tx Power (100%). MU-MIMO Activated to prevent bottlenecks. | ✅ Correct
Sec 04 | CSI Processed -> Predicted: 2 | STATUS: 2 USERS -> ACTION: Standard Tx Power (75%). Splitting spatial streams. | ✅ Correct
Sec 05 | CSI Processed -> Predicted: 1 | STATUS: 1 USER -> ACTION: Standard Tx Power (50%). Focusing beamforming on single target. | ✅ Correct
Sec 06 | CSI Processed -> Predicted: 3 | STATUS: CROWDED -> ACTION: MAX Tx